In [3]:
import os
from google.colab import userdata
os.environ['OPENAI_API_KEY'] = userdata.get('OPEN_AI')

In [2]:
# Install Essentail Libraries
!pip install -q youtube-transcript-api langchain-community langchain-openai faiss-cpu tiktoken python-dotenv

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.2/2.2 MB 23.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.5/2.5 MB 37.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 63.4/63.4 kB 3.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 31.3/31.3 MB 18.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 438.5/438.5 kB 20.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 44.4/44.4 kB 2.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 50.9/50.9 kB 3.4 MB/s eta 0:00:00


In [4]:
!pip install -qU langchain-community arxiv pymupdf

  Preparing metadata (setup.py) ... done
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 24.1/24.1 MB 32.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 81.3/81.3 kB 5.1 MB/s eta 0:00:00


In [38]:
from langchain_community.document_loaders import ArxivLoader

queries = ["Retrieval-Augmented Generation", "fine-tuning LLM", "RAG vs Fine-tuning"]
all_docs = []

In [39]:
for q in queries:
  loader = ArxivLoader(query=q, load_max_docs = 2)
  all_docs.append(loader.load())

MuPDF error: syntax error: could not parse color space (80 0 R)

MuPDF error: syntax error: could not parse color space (389 0 R)



In [41]:
# Limit to top 5 unique titles
unique_titles = set()
filtered_docs = []

for doc in all_docs:
    title = doc.metadata.get("Title", "").strip()
    if title and title not in unique_titles:
        unique_titles.add(title)
        filtered_docs.append(doc)
    if len(filtered_docs) == 5:
        break

In [52]:
# Making essential imports from these libraries
from langchain.text_splitter import RecursiveCharacterTextSplitter
from langchain_openai import OpenAIEmbeddings, ChatOpenAI
from langchain_community.vectorstores import FAISS
from langchain_core.prompts import PromptTemplate
from langchain_core.documents import Document

### Step 1: Indexing | Document Ingestion

In [53]:
# Step 1: Split the documents using RecursiveCharacterTextSplitter
text_splitter = RecursiveCharacterTextSplitter(
    chunk_size=500,
    chunk_overlap=100
)

In [55]:
split_docs = []
for doc in filtered_docs:
    chunks = text_splitter.split_text(doc.page_content)
    for chunk in chunks:
        split_docs.append(Document(page_content=chunk, metadata={"Title": doc.metadata.get("Title", "No Title")}))


In [108]:
len(split_docs)

1104

In [109]:
split_docs[0]

Document(metadata={'Title': 'Scaling Test-Time Inference with Policy-Optimized, Dynamic Retrieval-Augmented Generation via KV Caching and Decoding'}, page_content='arXiv:2504.01281v3  [cs.LG]  20 May 2025\nScaling Test-Time Inference with Policy-Optimized, Dynamic\nRetrieval-Augmented Generation via KV Caching and Decoding\nSakhinana Sagar Srinivas 1 Akash Das 1 Shivam Gupta 1 Venkataramana Runkana 1\nAbstract\nWe present a comprehensive framework for en-\nhancing Retrieval-Augmented Generation (RAG)\nsystems through dynamic retrieval strategies\nand reinforcement ﬁne-tuning.\nThis approach\nsigniﬁcantly improves large language models')

In [56]:
# Generate OpenAI embeddings
embeddings = OpenAIEmbeddings()


In [57]:
vectorstore = FAISS.from_documents(split_docs, embeddings)

In [58]:
# View a few ingested chunks
for i, doc in enumerate(split_docs[:5]):
    print(f"\n--- Chunk {i+1} ---")
    print("Title:", doc.metadata["Title"])
    print("Content:", doc.page_content[:300])


--- Chunk 1 ---
Title: Scaling Test-Time Inference with Policy-Optimized, Dynamic Retrieval-Augmented Generation via KV Caching and Decoding
Content: arXiv:2504.01281v3  [cs.LG]  20 May 2025
Scaling Test-Time Inference with Policy-Optimized, Dynamic
Retrieval-Augmented Generation via KV Caching and Decoding
Sakhinana Sagar Srinivas 1 Akash Das 1 Shivam Gupta 1 Venkataramana Runkana 1
Abstract
We present a comprehensive framework for en-
hancing R

--- Chunk 2 ---
Title: Scaling Test-Time Inference with Policy-Optimized, Dynamic Retrieval-Augmented Generation via KV Caching and Decoding
Content: and reinforcement ﬁne-tuning.
This approach
signiﬁcantly improves large language models
on knowledge-intensive tasks, including open-
domain question answering and complex rea-
soning. Our framework integrates two comple-
mentary techniques: Policy-Optimized Retrieval-
Augmented Generation (PORAG), 

--- Chunk 3 ---
Title: Scaling Test-Time Inference with Policy-Optimized, Dynamic Retrieval-Aug

In [61]:
# Embedding IDs
for k,v in vectorstore.index_to_docstore_id.items():
  if k>4:
    break
  print(k," : ",v)

0  :  1034b771-f880-49a4-acdc-d8bcbd352d57
1  :  1a9cbaab-8f4b-4eed-9393-dd07cac9b39e
2  :  e5b21bcc-dc40-4e46-b2d1-f72379a716c3
3  :  d8feeca9-ad06-4e9c-a73a-349a220a50b3
4  :  ea5121b8-38d9-488f-911d-25915ee7448b


In [62]:
# Let's check an embedding
vectorstore.get_by_ids(['d8feeca9-ad06-4e9c-a73a-349a220a50b3'])

[Document(id='d8feeca9-ad06-4e9c-a73a-349a220a50b3', metadata={'Title': 'Scaling Test-Time Inference with Policy-Optimized, Dynamic Retrieval-Augmented Generation via KV Caching and Decoding'}, page_content='cels in knowledge-intensive tasks, boosting out-\nput accuracy in RAG settings. We further pro-\npose CRITIC, a novel method to selectively com-\npress key-value caches by token importance, mit-\nigating memory bottlenecks in long-context ap-\nplications.\nThe framework also incorporates\ntest-time scaling techniques to dynamically bal-\nance reasoning depth and computational re-\nsources, alongside optimized decoding strategies\nfor faster inference. Experiments on benchmark')]

### Step 2: Retriever

In [64]:
## This retriever will first convert user query into embedding vector. Then, it will search the closest vectors in the vector DB. And it will return the those vectors.
retriever = vectorstore.as_retriever(search_type = 'mmr', search_kwargs = {"k":3})

In [66]:
query = "What are the pros and cons of RAG versus fine-tuning?"

In [69]:
retriever.invoke(query)

[Document(id='ef0eefb4-5bc7-4f75-b184-81e553d4400a', metadata={'Title': 'Scaling Test-Time Inference with Policy-Optimized, Dynamic Retrieval-Augmented Generation via KV Caching and Decoding'}, page_content='ing efﬁciency and performance.\nB. Comparing PORAG and RAFT\nMethodologies\nPolicy-Optimized\nRetrieval-Augmented\nGeneration\n(PORAG)\nand\nRetrieval-Augmented\nFine-Tuning\n(RAFT) (Zhang et al., 2024c) offer fundamentally dif-\nferent strategies for optimizing RAG systems.\nRAFT\nemploys supervised ﬁne-tuning (SFT) on static, curated\ndatasets containing predeﬁned question-response pairs\naccompanied by both relevant (“golden”) and irrelevant\n23'),
 Document(id='bf6b6f56-6db4-48a1-8c58-6a80cfd45776', metadata={'Title': 'When Scaling Meets LLM Finetuning: The Effect of Data, Model and Finetuning Method'}, page_content='Which finetuning method should we apply for a given task?\nUnfortunately, there is no universal\nanswer! Intuitively, there exists a critical point for finetuning 

### Step 3: Augmentation

In [68]:
prompt = PromptTemplate(
    template="""
    You are a helpful assistant.
    Answer only from the provided context transcript.
    If the context is insufficient, just say I don't know.

    {context}

    Question : {question}
    """,
    input_variables=['context', 'question']
)

In [102]:
question = "What's the difference between RAG and RAFT?"
retrieved_docs = retriever.invoke(question)

In [103]:
## Buiilding the relevant context
context_text = ""
for text in retrieved_docs:
  context_text += text.page_content

context_text

'ing efﬁciency and performance.\nB. Comparing PORAG and RAFT\nMethodologies\nPolicy-Optimized\nRetrieval-Augmented\nGeneration\n(PORAG)\nand\nRetrieval-Augmented\nFine-Tuning\n(RAFT) (Zhang et al., 2024c) offer fundamentally dif-\nferent strategies for optimizing RAG systems.\nRAFT\nemploys supervised ﬁne-tuning (SFT) on static, curated\ndatasets containing predeﬁned question-response pairs\naccompanied by both relevant (“golden”) and irrelevant\n23As for basic 2-shot RAG experiment on\nChatGPT-3.5, although there is an improvement\ncompared to the zero-shot baseline for both, the im-\nprovement on CMedQA is more pronounced. Case\nanalysis reveals that CMedQA’s corresponding cor-\npus is from question-and-answer pairs, whereas\nBioASQ consists of lengthy paragraphs, which\nleads to differences in passage format and retrieval\nquality. This may suggest two things: i. Simple\nRAG heavily relies on the retriever’s capability; ii.frozen. The RAFT loss function is deﬁned as:\nLRAFT(γ) = −E(

In [104]:
context = ""
for text in retrieved_docs:
  context += text.page_content + " "


In [105]:
final_prompt = prompt.invoke({'context':context, 'question':question})

In [106]:
context

'ing efﬁciency and performance.\nB. Comparing PORAG and RAFT\nMethodologies\nPolicy-Optimized\nRetrieval-Augmented\nGeneration\n(PORAG)\nand\nRetrieval-Augmented\nFine-Tuning\n(RAFT) (Zhang et al., 2024c) offer fundamentally dif-\nferent strategies for optimizing RAG systems.\nRAFT\nemploys supervised ﬁne-tuning (SFT) on static, curated\ndatasets containing predeﬁned question-response pairs\naccompanied by both relevant (“golden”) and irrelevant\n23 As for basic 2-shot RAG experiment on\nChatGPT-3.5, although there is an improvement\ncompared to the zero-shot baseline for both, the im-\nprovement on CMedQA is more pronounced. Case\nanalysis reveals that CMedQA’s corresponding cor-\npus is from question-and-answer pairs, whereas\nBioASQ consists of lengthy paragraphs, which\nleads to differences in passage format and retrieval\nquality. This may suggest two things: i. Simple\nRAG heavily relies on the retriever’s capability; ii. frozen. The RAFT loss function is deﬁned as:\nLRAFT(γ) = −

### Step 4: Generation

In [93]:
llm = ChatOpenAI(model='gpt-4o-mini', temperature = 0.5)

In [107]:
answer = llm.invoke(final_prompt)
print(answer.content)

RAG (Retrieval-Augmented Generation) is a general framework that combines retrieval and generation, while RAFT (Retrieval-Augmented Fine-Tuning) is a specific methodology that employs supervised fine-tuning on static, curated datasets with predefined question-response pairs. RAFT focuses on optimizing RAG systems through a defined loss function and utilizes both relevant and irrelevant documents during training, whereas RAG may not have the same structured training approach.


### Step 5: Building a Chain

In [110]:
## Import essential libraries
from langchain_core.runnables import RunnableParallel, RunnablePassthrough, RunnableLambda

In [111]:
def format_docs(retrived_docs):
  context_text = ""
  for text in retrieved_docs:
    context_text += text.page_content
  return context_text

In [112]:
parallel_chain = RunnableParallel({
    'context': retriever | RunnableLambda(format_docs),  # Retriever is pulling relevant docs and then we are formatting it into a single text
    'question': RunnablePassthrough()   # Question is input and output
})

In [113]:
parallel_chain.invoke('Who is Feroz?')

{'context': 'ing efﬁciency and performance.\nB. Comparing PORAG and RAFT\nMethodologies\nPolicy-Optimized\nRetrieval-Augmented\nGeneration\n(PORAG)\nand\nRetrieval-Augmented\nFine-Tuning\n(RAFT) (Zhang et al., 2024c) offer fundamentally dif-\nferent strategies for optimizing RAG systems.\nRAFT\nemploys supervised ﬁne-tuning (SFT) on static, curated\ndatasets containing predeﬁned question-response pairs\naccompanied by both relevant (“golden”) and irrelevant\n23As for basic 2-shot RAG experiment on\nChatGPT-3.5, although there is an improvement\ncompared to the zero-shot baseline for both, the im-\nprovement on CMedQA is more pronounced. Case\nanalysis reveals that CMedQA’s corresponding cor-\npus is from question-and-answer pairs, whereas\nBioASQ consists of lengthy paragraphs, which\nleads to differences in passage format and retrieval\nquality. This may suggest two things: i. Simple\nRAG heavily relies on the retriever’s capability; ii.frozen. The RAFT loss function is deﬁned as:\nLR

In [117]:
main_chain = parallel_chain | prompt | llm    # Connecting different segments

In [121]:
main_chain.invoke('Compare RAG and RAFT?').content

'RAG (Retrieval-Augmented Generation) and RAFT (Retrieval-Augmented Fine-Tuning) offer fundamentally different strategies for optimizing retrieval-augmented systems. RAFT employs supervised fine-tuning (SFT) on static, curated datasets that contain predefined question-response pairs, which are accompanied by both relevant ("golden") and irrelevant documents. The RAFT loss function is defined to optimize the model\'s ability to predict the correct output based on the input query and the retrieved documents.\n\nIn summary, RAG focuses on the retrieval and generation process, while RAFT emphasizes fine-tuning the model using curated datasets to improve performance on specific tasks.'

In [142]:
my_ques = ['Should I use RAG or RAFT?',
           'Is Fine-tuning better than RAG?',
           'How RAFT works?',
           'What is the formula of loss function of RAFT?',
           'Differentiate between PORAG vs RAFT']

In [141]:
main_chain.invoke().content

'Policy-Optimized Retrieval-Augmented Generation (PORAG) and Retrieval-Augmented Fine-Tuning (RAFT) are two different methodologies for optimizing Retrieval-Augmented Generation (RAG) systems. \n\nRAFT employs supervised fine-tuning (SFT) on static, curated datasets that contain predefined question-response pairs, including both relevant ("golden") and irrelevant responses. It uses a specific loss function to optimize the model\'s performance based on the retrieved documents and the input query.\n\nIn contrast, the context does not provide specific details about the PORAG methodology, making it difficult to fully differentiate between the two. However, it is clear that RAFT focuses on fine-tuning with curated datasets, while the details regarding PORAG\'s approach are not mentioned.'

In [143]:
llm_responses = []
rag_responese = []

In [144]:
for ques in my_ques:
  ans = llm.invoke(ques)
  llm_responses.append(str(ans.content))

  rag_responese.append(str(main_chain.invoke(ques).content))

In [145]:
i = 0
for ques in my_ques:
  print(ques)
  print()
  print('---LLM Response---')
  print(llm_responses[i])
  print()
  print('---RAG Response---')
  print(rag_responese[i])
  print('\n----------------------------\n')
  i+=1

Should I use RAG or RAFT?

---LLM Response---
The choice between RAG (Retrieval-Augmented Generation) and RAFT (Retrieval-Augmented Fine-Tuning) depends on your specific use case and requirements. Here’s a brief overview of both approaches to help you decide:

### RAG (Retrieval-Augmented Generation)
- **Purpose**: Combines retrieval of relevant documents with generative capabilities. It retrieves relevant information from a knowledge base and uses that information to generate responses.
- **Use Cases**: Ideal for tasks where you need to generate text based on external knowledge, such as question answering, summarization, or conversational agents.
- **Advantages**: 
  - Can leverage large external datasets to improve accuracy and relevance.
  - Produces more informed and contextually relevant responses.
- **Considerations**: Requires a good retrieval mechanism and may be computationally intensive.

### RAFT (Retrieval-Augmented Fine-Tuning)
- **Purpose**: Focuses on fine-tuning a pre-t

In [146]:
## Evaluation
!pip install rouge-score nltk


  Preparing metadata (setup.py) ... done
  Created wheel for rouge-score: filename=rouge_score-0.1.2-py3-none-any.whl size=24934 sha256=a3b10aabe4ff8c35be50954b1465b4d19d45bde54fe21bee93589660321cf82a
  Stored in directory: /root/.cache/pip/wheels/1e/19/43/8a442dc83660ca25e163e1bd1f89919284ab0d0c1475475148
Successfully built rouge-score


In [147]:
from nltk.translate.bleu_score import sentence_bleu, SmoothingFunction
from rouge_score import rouge_scorer
import pandas as pd


In [148]:
# Initialize metric tools
rouge = rouge_scorer.RougeScorer(['rougeL'], use_stemmer=True)
smooth = SmoothingFunction().method1

# Collect results
results = []

for i in range(len(my_ques)):
    bleu = sentence_bleu(
        [rag_responese[i].split()],
        llm_responses[i].split(),
        smoothing_function=smooth
    )
    rouge_l = rouge.score(rag_responese[i], llm_responses[i])['rougeL'].fmeasure

    results.append({
        "Question": my_ques[i],
        "BLEU (LLM vs RAG)": round(bleu, 3),
        "ROUGE-L (LLM vs RAG)": round(rouge_l, 3)
    })


In [149]:
df = pd.DataFrame(results)
df


,Question,BLEU (LLM vs RAG),ROUGE-L (LLM vs RAG)
0,Should I use RAG or RAFT?,0.000,0.000
1,Is Fine-tuning better than RAG?,0.000,0.000
2,How RAFT works?,0.002,0.077
3,What is the formula of loss function of RAFT?,0.008,0.096
4,Differentiate between PORAG vs RAFT,0.003,0.099


In [150]:
from sentence_transformers import SentenceTransformer, util
import torch


In [151]:
model = SentenceTransformer('all-MiniLM-L6-v2')  # Lightweight and accurate


modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/116 [00:00<?, ?B/s]

README.md:   0%|          | 0.00/10.5k [00:00<?, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/53.0 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/612 [00:00<?, ?B/s]

...

tokenizer_config.json:   0%|          | 0.00/350 [00:00<?, ?B/s]

vocab.txt:   0%|          | 0.00/232k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/466k [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/112 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

In [152]:
semantic_scores = []

for llm_resp, rag_resp in zip(llm_responses, rag_responese):
    llm_emb = model.encode(llm_resp, convert_to_tensor=True)
    rag_emb = model.encode(rag_resp, convert_to_tensor=True)

    cosine_sim = util.cos_sim(llm_emb, rag_emb).item()  # Extract scalar from tensor
    semantic_scores.append(round(cosine_sim, 3))


In [153]:
for i in range(len(results)):
    results[i]["Semantic Similarity"] = semantic_scores[i]

df = pd.DataFrame(results)
df


,Question,BLEU (LLM vs RAG),ROUGE-L (LLM vs RAG),Semantic Similarity
0,Should I use RAG or RAFT?,0.000,0.000,0.032
1,Is Fine-tuning better than RAG?,0.000,0.000,0.019
2,How RAFT works?,0.002,0.077,0.334
3,What is the formula of loss function of RAFT?,0.008,0.096,0.589
4,Differentiate between PORAG vs RAFT,0.003,0.099,0.279
